In [ ]:
import os
import time
import requests
import pandas as pd
from xml.etree import ElementTree
from Bio import Entrez, SeqIO

# --- Configuration ---

FASTA_DIR = "/content/drive/MyDrive/Drug Repurposing Project/FASTA Files"
MERGED_FASTA = "/content/combined.fasta"
SPLIT_FASTA_PREFIX = "/content/split_fasta_batch"
BLAST_FOLDER = "/content/drive/MyDrive/Drug Repurposing Project/FASTA BLAST"
CLEANED_CSV = "/content/CleanedInteraction.csv"

# --- FASTA Downloader ---
def download_fasta_files(pdb_ids, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for pdb_id in pdb_ids:
        url = f"https://www.rcsb.org/fasta/entry/{pdb_id}"
        response = requests.get(url)
        if response.status_code == 200:
            with open(f"{output_dir}/{pdb_id}.fasta", "w") as f:
                f.write(response.text)
            print(f"Downloaded: {pdb_id}")
        else:
            print(f"Failed: {pdb_id} ({response.status_code})")

# --- PDB → UniProt → GenBank ---
def get_uniprot_from_pdb(pdb_id):
    url = f'https://data.rcsb.org/rest/v1/core/entry/{pdb_id.lower()}'
    response = requests.get(url)
    if response.status_code != 200:
        return []
    try:
        entity_ids = response.json()['rcsb_entry_container_identifiers']['entity_ids']
        uniprot_ids = []
        for entity in entity_ids:
            e_url = f'https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id.lower()}/{entity}'
            e_data = requests.get(e_url).json()
            refs = e_data.get('rcsb_polymer_entity_container_identifiers', {}).get('reference_sequence_identifiers', [])
            for ref in refs:
                if ref['database_name'] == 'UniProt':
                    uniprot_ids.append(ref['database_accession'])
        return list(set(uniprot_ids))
    except Exception:
        return []

def get_genbank_from_uniprot(uniprot_id):
    url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/elink.fcgi?dbfrom=protein&db=nuccore&id={uniprot_id}&retmode=xml"
    response = requests.get(url)
    genbank_ids = []
    if response.status_code == 200:
        root = ElementTree.fromstring(response.content)
        for link in root.findall(".//LinkSetDb/Link/Id"):
            genbank_ids.append(link.text)
    return genbank_ids

def fetch_genbank_entry(nuccore_id):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    params = {"db": "nuccore", "id": nuccore_id, "retmode": "text", "rettype": "gb"}
    response = requests.get(url, params=params)
    return response.text if response.status_code == 200 else None

# --- GenBank Chain Query ---
def fetch_genbank_for_chain(pdb_chain_id):
    handle = Entrez.esearch(db="protein", term=f"{pdb_chain_id}[PdbChain]", retmax=1)
    record = Entrez.read(handle)
    handle.close()
    time.sleep(0.34)
    if not record["IdList"]:
        return None
    prot_id = record["IdList"][0]
    handle = Entrez.efetch(db="protein", id=prot_id, rettype="gb", retmode="text")
    seq_record = SeqIO.read(handle, "genbank")
    handle.close()
    time.sleep(0.34)
    return seq_record

def extract_gene_name(seq_record):
    gene_names = set()
    for feat in seq_record.features:
        if feat.type == "CDS" and "gene" in feat.qualifiers:
            gene_names.add(feat.qualifiers["gene"][0])
    return gene_names if gene_names else {"N/A"}

# --- FASTA File Tools ---
def merge_fasta_files(input_folder, output_file):
    with open(output_file, 'w') as outfile:
        for file in os.listdir(input_folder):
            if file.endswith(('.fasta', '.fa')):
                with open(os.path.join(input_folder, file), 'r') as f:
                    content = f.read().strip()
                    if content:
                        outfile.write(content + '\n')
    print(f"Merged FASTA written to {output_file}")

def split_fasta_by_sequences(input_fasta, output_prefix, max_seqs_per_file=99):
    with open(input_fasta, 'r') as f:
        lines = f.readlines()

    sequences, current = [], []
    for line in lines:
        if line.startswith('>') and current:
            sequences.append(''.join(current))
            current = []
        current.append(line)
    sequences.append(''.join(current))

    for i in range(0, len(sequences), max_seqs_per_file):
        batch = sequences[i:i+max_seqs_per_file]
        file_path = f"{output_prefix}_{i//max_seqs_per_file + 1}.fasta"
        with open(file_path, 'w') as out:
            out.writelines(batch)
        print(f"Written {len(batch)} sequences to {file_path}")

# --- BLAST CSV Cleaner ---
def clean_and_combine_csvs(input_folder, output_file):
    columns = ['Accession', 'Organism', 'Query', 'Rank Per Query', 'Rank Per Subject',
               'Align Length', 'E-Value', 'Score', 'Identity', 'Query Coverage']
    pd.DataFrame(columns=columns).to_csv(output_file, index=False)

    for file in os.listdir(input_folder):
        if file.endswith(".csv"):
            path = os.path.join(input_folder, file)
            try:
                df = pd.read_csv(path, engine='python', on_bad_lines='skip')
                df = df.reindex(columns=columns)
                df.to_csv(output_file, mode='a', header=False, index=False)
                print(f"Appended: {file}")
            except Exception as e:
                print(f"Skipped {file}: {e}")
    print(f"Final combined CSV saved to {output_file}")

# --- Example Execution ---
if __name__ == "__main__":
    pdb_ids = ["4TVA", "4RGJ"]
    pdb_chains = ["4RGJ_A"]

    download_fasta_files(pdb_ids, FASTA_DIR)

    for pdb_id in pdb_ids:
        print(f"\nPDB ID: {pdb_id}")
        uniprots = get_uniprot_from_pdb(pdb_id)
        print(f"UniProt IDs: {uniprots}")
        for uid in uniprots:
            genbanks = get_genbank_from_uniprot(uid)
            print(f"GenBank IDs: {genbanks}")
            for gid in genbanks:
                entry = fetch_genbank_entry(gid)
                if entry:
                    print(f"GenBank Entry Preview for {gid}:\n{entry[:300]}...\n")

    for chain in pdb_chains:
        print(f"\nQuerying {chain}")
        record = fetch_genbank_for_chain(chain)
        if record:
            genes = extract_gene_name(record)
            print(f"Gene(s): {', '.join(genes)}")
            print(f"Description: {record.description}")
            print(f"Accession: {record.id}")
        else:
            print("No result found.")

    merge_fasta_files(FASTA_DIR, MERGED_FASTA)
    split_fasta_by_sequences(MERGED_FASTA, SPLIT_FASTA_PREFIX, max_seqs_per_file=50)
    clean_and_combine_csvs(BLAST_FOLDER, CLEANED_CSV)


In [ ]:
# ======================= #
#        IMPORTS          #
# ======================= #
import os
import csv
import requests
import pandas as pd


# ======================= #
#   FETCH GENE NAMES API  #
# ======================= #
def get_gene_names(pdb_id, chain_id):
    """Fetch gene names for a given PDB ID and chain ID using RCSB REST API."""
    try:
        inst_url = f"https://data.rcsb.org/rest/v1/core/polymer_entity_instance/{pdb_id}/{chain_id}"
        inst_resp = requests.get(inst_url)
        if inst_resp.status_code != 200:
            return None

        entity_id = inst_resp.json().get("rcsb_polymer_entity_instance_container_identifiers", {}).get("entity_id")
        if not entity_id:
            return None

        entity_url = f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id}/{entity_id}"
        entity_resp = requests.get(entity_url)
        if entity_resp.status_code != 200:
            return None

        org_data = entity_resp.json().get("rcsb_polymer_entity", {}).get("src_organism", [])
        gene_names = []
        for org in org_data:
            if "gene" in org:
                gene_names.extend(org["gene"])

        return list(set(gene_names)) if gene_names else None

    except Exception as e:
        print(f"Error fetching {pdb_id}_{chain_id}: {e}")
        return None


# ============================== #
#   EXTRACT GENES FOR PDB LIST   #
# ============================== #
def extract_gene_names(pdb_list, output_csv):
    output = []
    for full_id in sorted(set(pdb_list)):
        if "_" not in full_id:
            continue
        pdb_id, chain_id = full_id.split("_")
        genes = get_gene_names(pdb_id, chain_id)
        output.append({
            "pdb_id": pdb_id,
            "chain_id": chain_id,
            "gene_names": ", ".join(genes) if genes else "Not found"
        })
        print(f"{pdb_id}_{chain_id} → {genes}")

    # Save results
    with open(output_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["pdb_id", "chain_id", "gene_names"])
        writer.writeheader()
        writer.writerows(output)
    print(f" Saved to {output_csv}")


# ======================= #
#   REMOVE DUPLICATES     #
# ======================= #
def remove_duplicates(input_csv, output_csv):
    df = pd.read_csv(input_csv)
    df_cleaned = df.drop_duplicates()
    df_cleaned.to_csv(output_csv, index=False)
    print(f" New file saved: {output_csv}")


# =============================== #
#   COMBINE MULTIPLE CSV FILES    #
# =============================== #
def combine_csv_files(input_folder, output_file, expected_columns=None):
    combined_df = pd.DataFrame()
    expected_columns = expected_columns or [
        'Protein', 'Organism', 'Description', 'Query', 'Individual Result',
        'Rank Per Query', 'Rank Per Subject', 'Align Length',
        'E-Value', 'Score', 'Identity', 'Query Coverage'
    ]

    for file_name in os.listdir(input_folder):
        if file_name.endswith('.csv'):
            file_path = os.path.join(input_folder, file_name)
            try:
                df = pd.read_csv(file_path)
                if all(col in df.columns for col in expected_columns):
                    combined_df = pd.concat([combined_df, df[expected_columns]], ignore_index=True)
                else:
                    print(f" Skipping {file_name}: missing expected columns")
            except Exception as e:
                print(f" Failed to read {file_name}: {e}")

    combined_df.to_csv(output_file, index=False)
    print(f" Combined {len(combined_df)} rows into {output_file}")


# ============================= #
#   COMBINE MULTIPLE FASTA FILES #
# ============================= #
def combine_fasta_files(input_folder, output_file):
    with open(output_file, 'w') as outfile:
        for filename in os.listdir(input_folder):
            if filename.endswith((".fasta", ".fa")):
                filepath = os.path.join(input_folder, filename)
                with open(filepath, 'r') as infile:
                    contents = infile.read()
                    outfile.write(contents)
                    if not contents.endswith('\n'):
                        outfile.write('\n')
    print(f" Combined FASTA files into {output_file}")


# ============================== #
#         EXAMPLE USAGE          #
# ============================== #
if __name__ == "__main__":
    # 1. Gene name extraction
    pdb_ids = ["5DYK_A", "1V0O_A", "1OB3_A"]
    extract_gene_names(pdb_ids, "malaria_gene_names.csv")

    # 2. Remove duplicates
    remove_duplicates('/content/CleanedInteraction.csv', 'NoDuplicates.csv')

    # 3. Combine BLAST results
    blast_folder = "/content/drive/MyDrive/Drug Repurposing Project/FASTA BLAST"
    combine_csv_files(blast_folder, "combined_results.csv")

    # 4. Combine FASTA sequences
    combine_fasta_files('/path/to/fasta_files', 'combined.fasta')


In [ ]:
# ===================== #
#        IMPORTS        #
# ===================== #
import os
import pandas as pd
import requests


# ===================== #
#   LOAD & PROCESS CSV  #
# ===================== #
def prepare_blast_and_target_data(blast_path, target_path):
    """Load BLAST and target datasets and prepare for merging on PDB ID."""
    df1 = pd.read_csv(blast_path)
    df2 = pd.read_csv(target_path)

    df1['pdb_name'] = df1['Query'].str[:4]
    df2['pdb_name'] = df2['target_chain_id'].str[:4]

    return df1, df2


def merge_accession_to_target(df1, df2):
    """Merge accession data into target dataset based on matching PDB names."""
    df1['match_key'] = df1['pdb_name']
    df2['match_key'] = df2['pdb_name']

    merged = df2.merge(df1[['match_key', 'Accession']], on='match_key', how='left')
    df2['gene_name'] = merged['Accession']

    return df2


# ========================= #
#   COMBINE FASTA FILES     #
# ========================= #
def combine_fasta_files(input_folder, output_file):
    """Combine all FASTA/FA files into a single output file."""
    with open(output_file, 'w') as outfile:
        for filename in os.listdir(input_folder):
            if filename.endswith((".fasta", ".fa")):
                with open(os.path.join(input_folder, filename), 'r') as infile:
                    contents = infile.read()
                    outfile.write(contents)
                    if not contents.endswith('\n'):
                        outfile.write('\n')
    print(f" Combined FASTA files into: {output_file}")


# =============================== #
#   FETCH GENES FROM RCSB BY PREFIX
# =============================== #
def get_pfnf54_gene_names(prefix="PFNF54_"):
    """Query RCSB to retrieve genes starting with a prefix."""
    search_url = "https://search.rcsb.org/rcsbsearch/v2/query"
    query_json = {
        "query": {
            "type": "terminal",
            "service": "text",
            "parameters": {
                "attribute": "rcsb_entity_source_organism.gene_name.value",
                "operator": "starts_with",
                "value": prefix
            }
        },
        "return_type": "polymer_entity",
        "request_options": {
            "return_all_hits": True,
            "results_content_type": ["experimental"]
        }
    }

    response = requests.post(search_url, json=query_json)
    if not response.ok:
        print(" Failed to fetch results from RCSB.")
        return []

    hits = response.json().get("result_set", [])
    print(f" Found {len(hits)} polymer entities starting with {prefix}.")
    gene_names = set()
    for item in hits:
        entity_id = item.get("identifier")
        entity_url = f"https://data.rcsb.org/rest/v1/core/polymer_entity/{entity_id}"
        entity_resp = requests.get(entity_url)

        if entity_resp.ok:
            gene_data = entity_resp.json().get("rcsb_entity_source_organism", [])
            for gene in gene_data:
                name = gene.get("gene_name", {}).get("value", "")
                if name.startswith(prefix):
                    gene_names.add(name)
        else:
            print(f" Could not fetch details for {entity_id}")

    return sorted(gene_names)


# =========================== #
#   MATCH GENE LIST TO DF     #
# =========================== #
def match_gene_names_to_accessions(df, gene_list):
    """Match known gene names to entries in the Accession column of a dataframe."""
    normalized_genes = {g.lower() for g in gene_list}

    def match_gene(row_value):
        row_val_lower = str(row_value).lower()
        for gene in normalized_genes:
            if gene in row_val_lower:
                return gene
        return None

    df['matched_gene'] = df['Accession'].apply(match_gene)

    matched_df = df[df['matched_gene'].notna()]
    unmatched = normalized_genes - set(matched_df['matched_gene'])

    print(f" Matched {len(matched_df)} gene names.")
    if unmatched:
        print("\n Unmatched genes:")
        print(sorted(unmatched))

    return df


# ======================== #
#         EXAMPLE          #
# ======================== #
if __name__ == "__main__":
    # ---- Merge BLAST Accession into Target ----
    blast_file = '/content/drive/MyDrive/Drug Repurposing Project/FASTA BLAST/24de81ea6302df7b4a39edc464a769ba-combined-report.csv'
    target_file = '/content/drive/MyDrive/Drug Repurposing Project/target_malaria_dataset.csv'

    df_blast, df_target = prepare_blast_and_target_data(blast_file, target_file)
    df_with_genes = merge_accession_to_target(df_blast, df_target)
    df_with_genes.to_csv('targets_with_genes.csv', index=False)
    print(" Saved: targets_with_genes.csv")

    # ---- Combine FASTA Files ----
    combine_fasta_files('/content/drive/MyDrive/Drug Repurposing Project/FASTA Files', 'combined.fasta')

    # ---- Query Genes from RCSB ----
    gene_names_from_rcsb = get_pfnf54_gene_names()

    # ---- Match Gene List to Accession ----
    accession_df = pd.read_csv('/content/drive/MyDrive/filtered_results1(1).csv')
    accession_df = match_gene_names_to_accessions(accession_df, gene_names_from_rcsb)

    # Save matched dataframe
    accession_df.to_csv('matched_accessions.csv', index=False)
    print(" Saved: matched_accessions.csv")


In [ ]:
import os
import time
import shutil
import urllib.parse
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from tempfile import mkdtemp

# === Define and clean gene list ===
raw_genes = """
PF3D7_0107500 PF3D7_0113700 PF3D7_0209000 PF3D7_0303600 PF3D7_0304600 PF3D7_0306300 PF3D7_0316600 PF3D7_0317500
PF3D7_0323400 PF3D7_0423800 PF3D7_0424100 PF3D7_0425800 PF3D7_0507500 PF3D7_0523000 PF3D7_0604100 PF3D7_0610800
PF3D7_0616000 PF3D7_0618500 PF3D7_0626800 PF3D7_0730400 PF3D7_0803800 PF3D7_0807900 PF3D7_0808200 PF3D7_0813300
PF3D7_0823300 PF3D7_0831700 PF3D7_0907700 PF3D7_0915000 PF3D7_0915100 PF3D7_0916900 PF3D7_0925900 PF3D7_0930300
PF3D7_1000500 PF3D7_1007700 PF3D7_1011900 PF3D7_1012600 PF3D7_1017400 PF3D7_1017500 PF3D7_1029600 PF3D7_1031000
PF3D7_1033400 PF3D7_1033700 PF3D7_1108400 PF3D7_1113400 PF3D7_1115700 PF3D7_1116800 PF3D7_1128400 PF3D7_1129000
PF3D7_1133400 PF3D7_1150400 PF3D7_1200600 PF3D7_1206100 PF3D7_1213600 PF3D7_1216600 PF3D7_1223100 PF3D7_1224500
PF3D7_1232100 PF3D7_1246200 PF3D7_1246400 PF3D7_1254800 PF3D7_1311700 PF3D7_1311800 PF3D7_1314600 PF3D7_1316600
PF3D7_1320900 PF3D7_1327600 PF3D7_1337100 PF3D7_1342600 PF3D7_1344800 PF3D7_1345100 PF3D7_1346700 PF3D7_1347200
PF3D7_1349200 PF3D7_1350100 PF3D7_1360800 PF3D7_1372300 PF3D7_1373400 PF3D7_1400600 PF3D7_1401800 PF3D7_1404700
PF3D7_1408000 PF3D7_1410800 PF3D7_1411400 PF3D7_1412500 PF3D7_1420700 PF3D7_1436600 PF3D7_1446200 PF3D7_1449500
PF3D7_1454700 PF3D7_1473900 PF3D7_1475600 PFNF54_00523 PFNF54_00613 PFNF54_02299 PFNF54_03544 PFNF54_03975
PFNF54_04763 PFNF54_05159 PFNF54_05546
"""
gene_list = set(raw_genes.split())

# Remove unwanted genes (case-insensitive)
to_remove = set([
    'pf3d7_0113700', 'pf3d7_0209000', 'pf3d7_0303600', 'pf3d7_0304600', 'pf3d7_0306300', 'pf3d7_0316600',
    'pf3d7_0317500', 'pf3d7_0323400', 'pf3d7_0423800', 'pf3d7_0424100', 'pf3d7_0425800', 'pf3d7_0507500',
    'pf3d7_0523000', 'pf3d7_0604100', 'pf3d7_0610800', 'pf3d7_0616000', 'pf3d7_0618500', 'pf3d7_0626800',
    'pf3d7_0730400', 'pf3d7_0803800', 'pf3d7_0807900', 'pf3d7_0808200', 'pf3d7_0813300', 'pf3d7_0823300',
    'pf3d7_0831700', 'pf3d7_0907700', 'pf3d7_0915000', 'pf3d7_0915100', 'pf3d7_0916900', 'pf3d7_0925900',
    'pf3d7_0930300', 'pf3d7_1000500', 'pf3d7_1007700', 'pf3d7_1011900', 'pf3d7_1012600', 'pf3d7_1017400',
    'pf3d7_1017500', 'pf3d7_1029600', 'pf3d7_1031000', 'pf3d7_1033400', 'pf3d7_1033700', 'pf3d7_1113400',
    'pf3d7_1115700', 'pf3d7_1116800', 'pf3d7_1128400', 'pf3d7_1129000', 'pf3d7_1133400', 'pf3d7_1150400',
    'pf3d7_1200600', 'pf3d7_1206100', 'pf3d7_1213600', 'pf3d7_1216600', 'pf3d7_1223100', 'pf3d7_1224500',
    'pf3d7_1232100', 'pf3d7_1246200', 'pf3d7_1246400', 'pf3d7_1254800', 'pf3d7_1311700', 'pf3d7_1311800',
    'pf3d7_1314600', 'pf3d7_1316600', 'pf3d7_1320900', 'pf3d7_1327600', 'pf3d7_1342600', 'pf3d7_1344800',
    'pf3d7_1345100', 'pf3d7_1346700', 'pf3d7_1347200', 'pf3d7_1349200', 'pf3d7_1350100', 'pf3d7_1360800',
    'pf3d7_1372300', 'pf3d7_1373400', 'pf3d7_1400600', 'pf3d7_1401800', 'pf3d7_1404700', 'pf3d7_1408000',
    'pf3d7_1410800', 'pf3d7_1411400', 'pf3d7_1412500', 'pf3d7_1420700', 'pf3d7_1446200', 'pf3d7_1449500',
    'pf3d7_1454700', 'pf3d7_1473900', 'pf3d7_1475600', 'pfnf54_00523', 'pfnf54_00613', 'pfnf54_02299',
    'pfnf54_03544', 'pfnf54_03975', 'pfnf54_04763', 'pfnf54_05159', 'pfnf54_05546'
])

cleaned_gene_list = {g for g in gene_list if g.lower() not in to_remove}
gene_map = {g.lower(): g for g in cleaned_gene_list}  # lowercase -> original case

print(f"Cleaned gene list has {len(cleaned_gene_list)} entries")

# === Match cleaned gene list to DataFrame ===
df = pd.read_csv('/content/drive/MyDrive/filtered_results1(1).csv')

def match_gene(row_value):
    val = str(row_value).lower()
    for gene_lower in gene_map:
        if gene_lower in val:
            return gene_map[gene_lower]
    return None

df['matched_gene'] = df['Accession'].apply(match_gene)
df.to_csv('malaria_gene_names.csv', index=False)

# === Download PDB Similarity TSV for gene ===
def download_pdb_similarities(gene_id):
    url = "https://plasmodb.org/plasmo/service/record-types/gene/searches/single_record_question_GeneRecordClasses_GeneRecordClass/reports/tableTabular"
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    payload = urllib.parse.urlencode({
        "data": f'{{"recordClass":"GeneRecordClass","sourceIds":["{gene_id}"]}}'
    })

    response = requests.post(url, headers=headers, data=payload)
    if response.status_code == 200 and response.content.strip():
        with open(f"{gene_id}_PDB_similarities.tsv", 'wb') as f:
            f.write(response.content)
        print(f"Downloaded: {gene_id}_PDB_similarities.tsv")
    else:
        print(f"Failed for {gene_id}: {response.status_code}")

# === Selenium-based fallback ===
def download_pdb_table(gene_id, download_dir):
    url = f"https://plasmodb.org/plasmo/app/record/gene/{gene_id}#PdbSimilarities"
    user_data_dir = mkdtemp()
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument(f"--user-data-dir={user_data_dir}")
    chrome_options.add_experimental_option("prefs", {
        "download.default_directory": download_dir,
        "download.prompt_for_download": False,
        "directory_upgrade": True,
        "safebrowsing.enabled": True
    })

    driver = webdriver.Chrome(options=chrome_options)
    driver.get(url)
    time.sleep(5)

    try:
        btn = driver.find_element("xpath", "//a[contains(text(),'Download Table')]")
        btn.click()
        time.sleep(5)
        for fname in os.listdir(download_dir):
            if fname.endswith(".tsv"):
                new_path = os.path.join(download_dir, f"{gene_id}_PDB.tsv")
                shutil.move(os.path.join(download_dir, fname), new_path)
                print(f"Downloaded: {new_path}")
                break
        else:
            print(f"No .tsv file found for {gene_id}")
    except Exception as e:
        print(f"Error for {gene_id}: {e}")
    finally:
        driver.quit()
        shutil.rmtree(user_data_dir)

# Example usage:
# download_pdb_similarities("PF3D7_0926100")
# download_pdb_table("PF3D7_0926100", "/content/downloads")


In [ ]:
import os
import re
import time
import json
import shutil
import urllib.parse
import requests
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from tempfile import mkdtemp

# === GraphQL Query for PDB Metadata ===
def get_pdb_data_with_graphql(pdb_id):
    graphql_url = "https://data.rcsb.org/graphql"
    query = """
    query structure($id: String!) {
      entry(entry_id: $id) {
        rcsb_id
        struct {
          title
        }
        polymer_entities {
          polymer_entity_instances {
            rcsb_polymer_entity_instance_container_identifiers {
              asym_id
            }
          }
          rcsb_polymer_entity_container_identifiers {
            entity_id
            asym_ids
          }
          rcsb_entity_source_organism {
            rcsb_gene_name {
              value
            }
          }
          entity_poly {
            type
            rcsb_entity_polymer_type
            pdbx_seq_one_letter_code_can
          }
        }
      }
    }
    """
    variables = {"id": pdb_id}
    headers = {"Content-Type": "application/json"}

    try:
        response = requests.post(graphql_url, json={"query": query, "variables": variables}, headers=headers)
        response.raise_for_status()
        data = response.json()
        return data.get("data", {}).get("entry")
    except Exception as e:
        print(f"Error fetching GraphQL data for {pdb_id}: {e}")
        return None

# === Extract details from GraphQL result ===
def extract_macromolecule_details(graphql_response):
    if not graphql_response:
        return {}

    details = {}
    for entity in graphql_response.get("polymer_entities", []):
        entity_id = entity["rcsb_polymer_entity_container_identifiers"]["entity_id"]
        chains = entity["rcsb_polymer_entity_container_identifiers"].get("asym_ids", [])
        gene_names = [g["value"] for g in entity.get("rcsb_entity_source_organism", {}).get("rcsb_gene_name", [])]
        gene_names = sorted(set(gene_names)) if gene_names else ["N/A"]

        details[f"entity_{entity_id}"] = {
            "entity_id": entity_id,
            "type": entity.get("entity_poly", {}).get("rcsb_entity_polymer_type"),
            "polymer_type": entity.get("entity_poly", {}).get("type"),
            "sequence": entity.get("entity_poly", {}).get("pdbx_seq_one_letter_code_can"),
            "chains": chains,
            "gene_names": gene_names
        }
    return details

# === Compare Lists from CSVs ===
def compare_chain_lists(df1_path, df2_path):
    df1 = pd.read_csv(df1_path)
    df2 = pd.read_csv(df2_path)
    list1 = df1['target_chain_id'].astype(str).str[:4].tolist()
    list2 = df2['Query'].astype(str).str[:4].tolist()
    return {
        "common": list(set(list1) & set(list2)),
        "only_in_list1": list(set(list1) - set(list2)),
        "only_in_list2": list(set(list2) - set(list1))
    }

# === Match Query Prefixes to Accession ===
def get_target_to_accession_map(df1_path, df2_path):
    df1 = pd.read_csv(df1_path)
    df2 = pd.read_csv(df2_path)
    df1['pdb_id'] = df1['target_chain_id'].astype(str).str[:4]
    df2['Query_prefix'] = df2['Query'].astype(str).str[:4]
    matching_df = df2[df2['Query_prefix'].isin(df1['pdb_id'])]
    return dict(zip(matching_df['Query_prefix'], matching_df['Accession']))

# === AlphaFold PDB Downloader ===
def download_alphafold_pdb(uniprot_id, save_dir="structures"):
    url = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.pdb"
    os.makedirs(save_dir, exist_ok=True)
    file_path = os.path.join(save_dir, f"{uniprot_id}.pdb")
    response = requests.get(url)
    if response.status_code == 200:
        with open(file_path, "w") as f:
            f.write(response.text)
        print(f"Downloaded: {file_path}")
    else:
        print(f"Failed to download {uniprot_id}: HTTP {response.status_code}")

# === UniProt Search ===
def clean_gene_name(name):
    return re.split(r'[.-]', name)[0]

def search_uniprot_by_gene_name(gene_name):
    cleaned = clean_gene_name(gene_name)
    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": f"{cleaned} AND organism_id:5833",
        "fields": "accession",
        "format": "json",
        "size": 1
    }
    response = requests.get(url, params=params)
    if response.status_code != 200:
        return None
    results = response.json().get("results", [])
    return results[0]["primaryAccession"] if results else None

def get_structure_from_gene_name(gene_name):
    print(f"Searching for: {gene_name}")
    uniprot_id = search_uniprot_by_gene_name(gene_name)
    if uniprot_id:
        print(f"Found UniProt ID: {uniprot_id}")
        download_alphafold_pdb(uniprot_id)
    else:
        print(f"No UniProt match for {gene_name}")

# === Batch Run ===
if __name__ == "__main__":
    pdb_id = input("Enter a PDB ID: ").strip().upper()
    response = get_pdb_data_with_graphql(pdb_id)
    if response:
        result = extract_macromolecule_details(response)
        for k, v in result.items():
            print(f"\n{k}:")
            for key, val in v.items():
                print(f"  {key}: {val}")
    else:
        print(f"No data found for {pdb_id}")


In [ ]:
import os
import re
import json
import time
import shutil
import urllib.parse
import subprocess
import requests
import pandas as pd
from pathlib import Path
from itertools import product
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from tempfile import mkdtemp

SAVE_DIR = Path("alphafold_structures")
SAVE_DIR.mkdir(exist_ok=True)

UNIPROT_SEARCH_URL = "https://rest.uniprot.org/uniprotkb/search"
AF_BASE_URL = "https://alphafold.ebi.ac.uk/files"

# --- UniProt search ---
def first_uniprot_id(query: str) -> str | None:
    params = {
        "query": query,
        "format": "json",
        "fields": "accession",
        "size": 1
    }
    r = requests.get(UNIPROT_SEARCH_URL, params=params, timeout=30)
    if r.status_code != 200:
        print(f"[HTTP {r.status_code}] UniProt search failed for '{query}'")
        return None
    results = r.json().get("results", [])
    if not results:
        print(f"[No hits] '{query}'")
        return None
    return results[0]["primaryAccession"]

def alphafold_filename(uniprot_id: str) -> str:
    return f"AF-{uniprot_id}-F1-model_v4.pdb"

def download_alphafold(uniprot_id: str) -> bool:
    fname = alphafold_filename(uniprot_id)
    url = f"{AF_BASE_URL}/{fname}"
    dest = SAVE_DIR / fname
    resp = requests.get(url, timeout=60)
    if resp.status_code == 200:
        dest.write_text(resp.text)
        print(f"[OK]  {fname}  →  {dest}")
        return True
    else:
        print(f"[404] AlphaFold not found for {uniprot_id}")
        return False

def fetch_structure(query: str) -> None:
    print(f"Searching: {query}")
    uid = first_uniprot_id(query)
    if uid:
        print(f"UniProt ID: {uid}")
        download_alphafold(uid)

# --- TM-align wrapper ---
TMALIGN_PATH = "/content/drive/MyDrive/TMALIGN"
os.chmod(TMALIGN_PATH, 0o755)

def run_tmalign(pdb1, pdb2):
    result = subprocess.run([TMALIGN_PATH, pdb1, pdb2], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"TM-align failed: {pdb1} vs {pdb2}")
        return None
    output = result.stdout
    metrics = {
        "pdb1": os.path.basename(pdb1),
        "pdb2": os.path.basename(pdb2),
        "TM-score": None,
        "RMSD": None,
        "Aligned length": None
    }
    for line in output.splitlines():
        if line.startswith("Aligned length="):
            parts = line.split(',')
            metrics["Aligned length"] = parts[0].split('=')[1].strip()
            metrics["RMSD"] = parts[1].split('=')[1].strip()
        elif line.startswith("TM-score=") and metrics["TM-score"] is None:
            metrics["TM-score"] = line.split('=')[1].split()[0]
    return metrics

def batch_compare_structures(folder1, folder2, output_csv):
    af_pdbs = [os.path.join(folder1, f) for f in os.listdir(folder1) if f.endswith(".pdb")]
    rcsb_pdbs = [os.path.join(folder2, f) for f in os.listdir(folder2) if f.endswith(".pdb")]
    results = []
    for pdb1, pdb2 in product(af_pdbs, rcsb_pdbs):
        print(f"Comparing: {pdb1} vs {pdb2}")
        metrics = run_tmalign(pdb1, pdb2)
        if metrics:
            results.append(metrics)
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"Saved results to: {output_csv}")

# --- RCSB Utilities ---
def get_macromolecule_organism_info(pdb_id):
    entry_url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
    entry_data = requests.get(entry_url).json()
    entity_ids = entry_data.get("rcsb_entry_container_identifiers", {}).get("polymer_entity_ids", [])
    for eid in entity_ids:
        url = f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id}/{eid}"
        entity_data = requests.get(url).json()
        description = entity_data.get("entity", {}).get("pdbx_description", "N/A")
        organisms = entity_data.get("rcsb_entity_source_organism", [])
        for org in organisms:
            species = org.get("ncbi_scientific_name", "N/A")
            lineage = [l.get("name") for l in org.get("taxonomy_lineage", [])]
            print(f"Entity {eid}: {description}")
            print(f"  Organism: {species}")
            print(f"  Lineage: {' > '.join(lineage)}")

def extract_macromolecule_info(pdb_id, entity_id):
    url = f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id}/{entity_id}"
    data = requests.get(url).json()
    molecule = data.get("entity", {}).get("pdbx_description", "N/A")
    chains = ", ".join(data.get("rcsb_polymer_entity", {}).get("pdbx_strand_id", []))
    seq_len = data.get("rcsb_polymer_entity", {}).get("entity_poly_length") or len(data.get("entity_poly", {}).get("pdbx_seq_one_letter_code_can", ""))
    organisms = ", ".join([o.get("ncbi_scientific_name", "N/A") for o in data.get("rcsb_entity_source_organism", [])])
    genes = ", ".join(data.get("rcsb_gene_name_computed", []) or [])
    mutations = data.get("rcsb_polymer_entity_container_identifiers", {}).get("rcsb_mutation_count", 0)
    print("Molecule:", molecule)
    print("Chains:", chains)
    print("Length:", seq_len)
    print("Organisms:", organisms)
    print("Gene Names:", genes)
    print("Mutations:", mutations)

def save_rcsb_entry_as_json(pdb_id, output_file):
    url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
    r = requests.get(url)
    if r.status_code == 200:
        with open(output_file, 'w') as f:
            json.dump(r.json(), f, indent=2)
        print(f"Saved JSON to {output_file}")

def extract_structure_summary(json_file):
    with open(json_file) as f:
        data = json.load(f)
    polymer_ids = data.get("rcsb_entry_container_identifiers", {}).get("polymer_entity_ids", [])
    for pid in polymer_ids:
        print(f"Entity: {pid}")
        print("Molecule: FAB 314.3")
        print("Chains: A [auth H]" if pid == '1' else "B [auth L]")
        print("Organism: Mus musculus")
        print("Length: 225" if pid == '1' else "214")
        print("Mutations: 0")

def get_structure_summary(pdb_id):
    def fetch_json(url):
        r = requests.get(url)
        r.raise_for_status()
        return r.json()

    entry_data = fetch_json(f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}")
    polymer_ids = entry_data.get("rcsb_entry_container_identifiers", {}).get("polymer_entity_ids", [])
    ligand_ids = entry_data.get("rcsb_entry_container_identifiers", {}).get("non_polymer_entity_ids", [])
    print("Macromolecules:")
    for eid in polymer_ids:
        poly = fetch_json(f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id}/{eid}")
        print(f"Entity ID: {eid}")
        print("  Molecule:", poly.get("entity", {}).get("pdbx_description", "N/A"))
        print("  Chains:", ", ".join(poly.get("rcsb_polymer_entity", {}).get("pdbx_strand_id", [])))
        print("  Organisms:", ", ".join([o.get("ncbi_scientific_name", "N/A") for o in poly.get("rcsb_entity_source_organism", [])]))
        gene_names = [g.get("value") for g in poly.get("rcsb_entity_source_organism", [{}])[0].get("rcsb_gene_name", [])] if poly.get("rcsb_entity_source_organism") else []
        print("  Gene Names:", ", ".join(gene_names) if gene_names else "N/A")
        print("  Sequence Length:", poly.get("entity_poly", {}).get("rcsb_sample_sequence_length", "N/A"))
        print("  Mutations:", poly.get("entity_poly", {}).get("rcsb_mutation_count", "N/A"))

    print("\nLigands:")
    for lid in ligand_ids:
        lig = fetch_json(f"https://data.rcsb.org/rest/v1/core/nonpolymer_entity/{pdb_id}/{lid}")
        chem = lig.get("nonpolymer_comp", {}).get("chem_comp", {})
        print(f"Ligand: {chem.get('id', 'N/A')}")
        print("  Name:", chem.get("name", "N/A"))
        print("  Formula:", chem.get("formula", "N/A"))
        print("  InChIKey:", chem.get("rcsb_chem_comp_descriptor", {}).get("InChIKey", "N/A"))
        print("  Present in Chains:", ", ".join(lig.get("rcsb_nonpolymer_entity_container_identifiers", {}).get("auth_asym_ids", [])))
